# Experiment 6 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 6/_analysis/` and compares corrupted runs with the Experiment 4 baseline CSV.

The notebook does not read raw experiment output folders.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

POLYMER_CATEGORIES = ["Polymer_sheet"]

SEVERITIES = [1, 2, 3]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 6"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
EXP_CSV = ANALYSIS_DIR / "experiment6_runs_cache.csv"
BASELINE_CSV = ANALYSIS_DIR / "experiment4_polymer_baseline_cache.csv"

print(f"Runs CSV: {EXP_CSV}")
print(f"Baseline CSV: {BASELINE_CSV}")


In [ ]:
df = pd.read_csv(EXP_CSV)
baseline_df = pd.read_csv(BASELINE_CSV)
normalize_latency(df)
normalize_latency(baseline_df)
if not df.empty:
    df["severity"] = df["severity"].astype(int)

CORRUPTIONS = sorted(df["corruption"].dropna().unique()) if not df.empty else []
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated corrupted runs across models {MODELS_PRESENT} and corruptions {CORRUPTIONS}.")
print(f"Loaded {len(baseline_df)} consolidated baseline rows (models: {sorted(baseline_df['model'].dropna().unique()) if not baseline_df.empty else []}).")


## 1. Runs per category x model - one table per corruption

For each corruption kind, the count of run folders found in `Experiment 6/`. Each cell encodes severity counts in the layout:

| s1 (top-left) | s2 (top-right) |
| :---: | :---: |
| s3 (bottom-center, spans) ||

In [ ]:
def _fmt_or_dash(v, fmt) -> str:
    if v is None or pd.isna(v):
        return "\u2014"
    return fmt(v)


def severity_cell_html(values: dict, fmt=lambda v: f"{v}", color_by_sign: bool = False) -> str:
    def style(v):
        if not color_by_sign or v is None or pd.isna(v):
            return ""
        if v < 0:
            return "color:#b00020"
        if v > 0:
            return "color:#0a7d2c"
        return "color:#444"

    s1, s2, s3 = values.get(1), values.get(2), values.get(3)
    return (
        "<table style='border-collapse:collapse;font-size:10px;width:100%;table-layout:fixed'>"
        "<tr>"
        f"<td style='padding:1px 3px;border-right:1px solid #ddd;border-bottom:1px solid #ddd;text-align:center'>"
        f"<span style='color:#888'>s1</span><br><b style='{style(s1)}'>{_fmt_or_dash(s1, fmt)}</b></td>"
        f"<td style='padding:1px 3px;border-bottom:1px solid #ddd;text-align:center'>"
        f"<span style='color:#888'>s2</span><br><b style='{style(s2)}'>{_fmt_or_dash(s2, fmt)}</b></td>"
        "</tr><tr>"
        f"<td colspan='2' style='padding:1px 3px;text-align:center'>"
        f"<span style='color:#888'>s3</span><br><b style='{style(s3)}'>{_fmt_or_dash(s3, fmt)}</b></td>"
        "</tr></table>"
    )


def render_severity_grid(
    df_in: pd.DataFrame,
    *,
    row_axis: str,
    row_values: list,
    col_axis: str,
    col_values: list,
    metric: str,
    aggregate: str = "mean",
    fmt=lambda v: f"{v:.3f}",
    title: str | None = None,
) -> str:
    grouped = df_in.groupby([row_axis, col_axis, "severity"])[metric]
    series = grouped.size() if aggregate == "count" else grouped.mean()

    def get(r, c, s):
        try:
            return float(series.loc[(r, c, s)])
        except KeyError:
            return 0.0 if aggregate == "count" else float("nan")

    header = "".join(
        f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{c}</th>"
        for c in col_values
    )
    body = []
    for r in row_values:
        cells = "".join(
            f"<td style='border:1px solid #999;padding:2px;vertical-align:top'>"
            f"{severity_cell_html({s: get(r, c, s) for s in SEVERITIES}, fmt=fmt)}</td>"
            for c in col_values
        )
        body.append(
            f"<tr><th style='border:1px solid #999;padding:4px;text-align:left;background:#f3f3f3'>{r}</th>{cells}</tr>"
        )
    title_html = f"<h4 style='margin:8px 0 4px'>{title}</h4>" if title else ""
    return (
        title_html
        + "<table style='border-collapse:collapse'>"
        + f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{row_axis}</th>{header}</tr>"
        + "".join(body)
        + "</table>"
    )


for corr in CORRUPTIONS:
    sub = df[df["corruption"] == corr]
    html = render_severity_grid(
        sub,
        row_axis="category", row_values=POLYMER_CATEGORIES,
        col_axis="model", col_values=ALL_MODELS,
        metric="experiment", aggregate="count",
        fmt=lambda v: f"{int(v)}",
        title=f"Runs \u2014 corruption: {corr}",
    )
    display(HTML(html))

## 2. Per-model averaged metrics - one table per corruption

For each corruption, the mean of each metric over the categories run, broken down by severity. The cell layout is the same s1/s2/s3 schema.

In [ ]:
METRICS_AVG = ["auroc", "aupr", "precision", "recall", "mean_latency_ms", "f1"]


def render_metric_x_model_table(sub: pd.DataFrame, title: str) -> str:
    parts = [f"<h4 style='margin:8px 0 4px'>{title}</h4>"]
    header = "".join(
        f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{m}</th>"
        for m in ALL_MODELS
    )
    parts.append(
        "<table style='border-collapse:collapse'>"
        f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3'>metric</th>{header}</tr>"
    )
    for metric in METRICS_AVG:
        cells = []
        is_lat = metric == "mean_latency_ms"
        fmt = (lambda v: f"{v:.1f}") if is_lat else (lambda v: f"{v:.3f}")
        for model in ALL_MODELS:
            sub_m = sub[sub["model"] == model]
            values = {}
            for s in SEVERITIES:
                slice_ = sub_m[sub_m["severity"] == s][metric]
                values[s] = float(slice_.mean()) if not slice_.empty else float("nan")
            cells.append(
                "<td style='border:1px solid #999;padding:2px;vertical-align:top'>"
                f"{severity_cell_html(values, fmt=fmt)}</td>"
            )
        parts.append(
            f"<tr><th style='border:1px solid #999;padding:4px;text-align:left;background:#f3f3f3'>{metric}</th>{''.join(cells)}</tr>"
        )
    parts.append("</table>")
    return "".join(parts)


for corr in CORRUPTIONS:
    sub = df[df["corruption"] == corr]
    display(HTML(render_metric_x_model_table(sub, f"Metrics \u2014 corruption: {corr}")))

## 3. Degradation mini-matrix per corruption (category x model)

Each cell shows **Delta AUROC = AUROC_corrupted - AUROC_baseline** at each severity, using the same s1/s2/s3 schema. Baseline is the matching `(model, category)` run from `Experiment 4` (uncorrupted). Red = degradation, green = improvement, dash = baseline or corrupted run missing.

This view answers: *for a given (model, category) cell, how does this corruption degrade AUROC as severity grows-*

In [ ]:
baseline_idx = (
    baseline_df.set_index(["model", "category"])
    if not baseline_df.empty
    else pd.DataFrame()
)


def baseline_value(model: str, category: str, metric: str) -> float:
    if baseline_idx.empty:
        return float("nan")
    try:
        v = baseline_idx.loc[(model, category)][metric]
        return float(v) if pd.notna(v) else float("nan")
    except KeyError:
        return float("nan")


def render_degradation_grid(metric: str, fmt=lambda v: f"{v:+.3f}") -> None:
    models_to_show = MODELS_PRESENT if MODELS_PRESENT else ALL_MODELS
    for corr in CORRUPTIONS:
        sub = df[df["corruption"] == corr]
        idx = sub.set_index(["model", "category", "severity"])[metric]
        header = "".join(
            f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3'>{m}</th>"
            for m in models_to_show
        )
        rows_html = []
        for cat in POLYMER_CATEGORIES:
            cells = []
            for model in models_to_show:
                base = baseline_value(model, cat, metric)
                deltas = {}
                for s in SEVERITIES:
                    try:
                        cur = float(idx.loc[(model, cat, s)])
                    except KeyError:
                        cur = float("nan")
                    if pd.isna(cur) or pd.isna(base):
                        deltas[s] = float("nan")
                    else:
                        deltas[s] = cur - base
                cells.append(
                    "<td style='border:1px solid #999;padding:2px;vertical-align:top'>"
                    f"{severity_cell_html(deltas, fmt=fmt, color_by_sign=True)}</td>"
                )
            rows_html.append(
                f"<tr><th style='border:1px solid #999;padding:4px;text-align:left;background:#f3f3f3'>{cat}</th>{''.join(cells)}</tr>"
            )
        display(HTML(
            f"<h4 style='margin:8px 0 4px'>\u0394{metric} (corrupted \u2212 baseline) \u2014 corruption: {corr}</h4>"
            + "<table style='border-collapse:collapse'>"
            + f"<tr><th style='border:1px solid #999;padding:4px;background:#f3f3f3'>category</th>{header}</tr>"
            + "".join(rows_html)
            + "</table>"
        ))


render_degradation_grid("auroc")

## 4. Per-(model, corruption) detailed degradation table

One block per `(model, corruption)`. Each row is a category, and for each headline metric (AUROC, F1, AUPR) we show:

- `base` - the uncorrupted Experiment 4 value;
- `s<k>` - the corrupted value at severity *k*;
- `delta_s<k>` - corrupted minus baseline (red = drop, green = gain).

This is the granular companion to section 3 - useful when you want exact numbers and per-metric drops, not just an AUROC overview.

In [ ]:
METRICS_DETAIL = ["auroc", "f1", "aupr"]


def _delta_style(v: float) -> str:
    if pd.isna(v):
        return ""
    if v < 0:
        return "color:#b00020;font-weight:bold"
    if v > 0:
        return "color:#0a7d2c;font-weight:bold"
    return ""


def render_detail_table(model: str, corr: str) -> str:
    sub = df[(df["model"] == model) & (df["corruption"] == corr)]
    if sub.empty:
        return ""
    categories_run = sorted(sub["category"].unique())
    headers = ["category"]
    for metric in METRICS_DETAIL:
        headers.append(f"base_{metric}")
        for s in SEVERITIES:
            headers.append(f"s{s}_{metric}")
            headers.append(f"\u0394s{s}")
    header_html = "".join(
        f"<th style='border:1px solid #999;padding:4px;background:#f3f3f3;font-size:11px;text-align:center'>{h}</th>"
        for h in headers
    )
    fmt3 = lambda v: f"{v:.3f}"
    fmtd = lambda v: f"{v:+.3f}"
    rows_html = []
    for cat in categories_run:
        cells = [f"<td style='border:1px solid #999;padding:3px;text-align:left'><b>{cat}</b></td>"]
        for metric in METRICS_DETAIL:
            base_v = baseline_value(model, cat, metric)
            cells.append(
                f"<td style='border:1px solid #999;padding:3px;text-align:right;color:#555;font-size:11px'>{_fmt_or_dash(base_v, fmt3)}</td>"
            )
            for s in SEVERITIES:
                row = sub[(sub["category"] == cat) & (sub["severity"] == s)][metric]
                cur_v = float(row.iloc[0]) if not row.empty else float("nan")
                delta = (cur_v - base_v) if (pd.notna(cur_v) and pd.notna(base_v)) else float("nan")
                cells.append(
                    f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px'>{_fmt_or_dash(cur_v, fmt3)}</td>"
                )
                cells.append(
                    f"<td style='border:1px solid #999;padding:3px;text-align:right;font-size:11px;{_delta_style(delta)}'>{_fmt_or_dash(delta, fmtd)}</td>"
                )
        rows_html.append(f"<tr>{''.join(cells)}</tr>")
    return (
        f"<h4 style='margin:8px 0 4px'>{model} \u2014 corruption: {corr} ({len(categories_run)} categories)</h4>"
        "<table style='border-collapse:collapse'>"
        f"<tr>{header_html}</tr>"
        + "".join(rows_html)
        + "</table>"
    )


for model in MODELS_PRESENT:
    for corr in CORRUPTIONS:
        html = render_detail_table(model, corr)
        if html:
            display(HTML(html))

## 5. Degradation heatmaps - small multiples per corruption

For each corruption, one subplot per model present in Experiment 6. Inside each subplot: rows = categories, columns = severities (s1, s2, s3), values = Delta metric (corrupted - baseline). Diverging colormap (`RdBu`): red = degradation, blue = improvement, grey = missing data.

Use these to spot:

- which **corruption kinds** hit a model hardest (compare across the per-corruption figures);
- which **categories** are sensitive (look for solid red rows);
- how degradation **escalates with severity** (read each row left-to-right inside a subplot).

In [ ]:
def degradation_heatmaps(metric: str = "auroc", *, vlim: float = 0.5) -> None:
    if not MODELS_PRESENT:
        print("No corrupted runs loaded - nothing to plot.")
        return
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad(color="#e5e5e5")
    for corr in CORRUPTIONS:
        ncols = len(MODELS_PRESENT)
        fig, axes = plt.subplots(
            1, ncols,
            figsize=(ncols * (0.7 * len(SEVERITIES) + 2.0),
                     0.32 * len(POLYMER_CATEGORIES) + 1.8),
            squeeze=False,
        )
        fig.suptitle(f"\u0394{metric} vs Experiment 4 baseline \u2014 corruption: {corr}", fontsize=12)
        for j, model in enumerate(MODELS_PRESENT):
            ax = axes[0][j]
            mat = np.full((len(POLYMER_CATEGORIES), len(SEVERITIES)), np.nan)
            for ri, cat in enumerate(POLYMER_CATEGORIES):
                base = baseline_value(model, cat, metric)
                for ci, s in enumerate(SEVERITIES):
                    row = df[(df["model"] == model) & (df["corruption"] == corr) & (df["category"] == cat) & (df["severity"] == s)]
                    if row.empty:
                        continue
                    cur = float(row[metric].iloc[0])
                    if pd.notna(cur) and pd.notna(base):
                        mat[ri, ci] = cur - base
            masked = np.ma.masked_invalid(mat)
            im = ax.imshow(masked, aspect="auto", cmap=cmap, vmin=-vlim, vmax=vlim)
            ax.set_xticks(range(len(SEVERITIES)))
            ax.set_xticklabels([f"s{s}" for s in SEVERITIES])
            ax.set_yticks(range(len(POLYMER_CATEGORIES)))
            ax.set_yticklabels(POLYMER_CATEGORIES, fontsize=8)
            ax.set_title(model, fontsize=10)
            for ri in range(mat.shape[0]):
                for ci in range(mat.shape[1]):
                    v = mat[ri, ci]
                    if pd.notna(v):
                        ax.text(ci, ri, f"{v:+.2f}", ha="center", va="center",
                                fontsize=7, color="black")
            fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
        plt.close(fig)


degradation_heatmaps("auroc")
degradation_heatmaps("f1")